## TuRBO

### 1. Introdução

O Trust Region Bayesian Optimization (TuRBO) foi proposto em 2019 pela Uber, em que ocorrem otimizações locais simultâneas usando modelos probabilísticos independentes. Cada um dos modelos locais aproveitam a vantagem do BO (robustez para ruído e estimativa da incerteza), enquanto permite modelagem heterogênea.

### 2. O algoritmo

Precisamos encontrar um $x^*$ tal que $x^* \in \Omega$ e $f(x^*) \leq f(x), \space \forall x \in \Omega$

Ou seja, desejamos encontrar um $x^*$, no domínio, que é mínimo da função para todo o conjunto imagem. Usaremos $f: \Omega \to \mathbb{R}$ e $\Omega = [0, 1]^d$. A limitação de a função ter valores somente entre 0 e 1 em cada dimensão pode ser contornada ao aplicar uma transformação linear na função, para a limitar nesse intervalo.

Vejamos primeiro como funciona cada tentativa local, para após entender como operam simultâneamente

#### 2.1 Trust Region

Escolhe-se como Trust Region um hiperretângulo centrado na melhor solução até o momento ($x^*$) ou, para dados com ruído, o ponto de menor média após aplicar o processo gaussiano (GP). Inicialize-se o retângulo com lados lados $L_{init}$ em cada dimensão, que será atualizado para:

$$
L_i = \frac{\lambda_iL}{(\prod_{j=1}^d\lambda_j)^{1/d}}
$$

Em que o lambda é o lengthscale do GP na dimensão. Isso é importante, pois se o $\lambda$ for grande, significa que está variando lentamente na região, podendo avançar mais, ou o contrário se $\lambda$ for pequeno. Além disso, a equação mantém o volume total como $L^{d}_{init}$.

É usado agora uma aquisition function para eleger no espaço da Trust Region q pontos possíveis, objetivando-se resolver o subproblema dentro da região. Para isso, é usado um algoritmo em que define-se sucesso se o ponto conseguiu diminuir a função, enquanto fracasso se não conseguiu. Caso o modelo atinja um número arbitrário definido de sucessos, o lado L deverá dobrar e tentar de novo. Caso atinja o número de fracassos, os lados são divididos por 2. Se o lado encolher para menor que uma marca arbitrária $L_{min}$ anteriormente definida, os lados são reinicializados para $L_{init}$, e o mesmo se os lados passarem de $L_{max}$

Dessa forma, consegue-se manter uma trust region eficaz e explorar os menores valores possíveis com ela.

#### 2.2 TuRBO

Porém, o que acontece no TuRBO é que são geradas m trust regions simultâneamente, começadas de pontos diferentes. Assim, o TuRBO precisa escolher um conjunto de q pontos para avaliar, mas esses pontos podem vir de qualquer uma das m trust regions ativas. O desafio é decidir, ao mesmo tempo, de qual região explorar e qual ponto específico dentro dela escolher, sem tratar essas duas decisões separadamente.

Para resolver isso, o modelo usa Thompson Sampling (TS) de forma global. Nele, em vez de escolher primeiro uma trust region e depois otimizar dentro dela, o TuRBO faz com que todas as trust regions compitam entre si a cada ponto selecionado.

Para selecionar o i-ésimo ponto do batch, o algoritmo começa amostrando, em cada trust region, uma função possível a partir do modelo GP local dessa região. Essa função amostrada representa uma hipótese plausível de como a função objetivo se comporta naquela região, dado o que já foi observado. Em seguida, dentro de cada trust region, ele resolve um problema de otimização local: encontra o ponto que minimiza essa função amostrada dentro dos limites da trust region.

Nesse momento, cada trust region produziu exatamente um candidato, que é o melhor ponto segundo a função que ela “imaginou”. O próximo passo é comparar esses candidatos entre todas as trust regions. O algoritmo então escolhe aquele cujo valor (na função amostrada correspondente) é o menor entre todos — ou seja, o ponto que parece mais promissor considerando todas as regiões simultaneamente.

Esse processo gera um único ponto do batch. Para construir o batch completo de tamanho q, o procedimento é repetido q vezes, e a cada repetição novas funções são amostradas dos GPs. Isso faz com que a escolha de pontos seja naturalmente estocástica e balanceie exploração e exploração: trust regions mais promissoras tendem a produzir melhores candidatos com mais frequência, mas regiões menos exploradas ainda têm chance de serem escolhidas devido à incerteza capturada pelo GP.

Assim, o TuRBO evita ter que decidir explicitamente qual trust region explorar. Em vez disso, essa decisão emerge automaticamente do processo de amostragem e competição entre regiões, permitindo que o algoritmo distribua seu orçamento de avaliações de forma adaptativa e eficiente.

#### Teste de resolução da Função de Ackley (D=20) com o uso do algoritmo TuRBO, conforme proposto no tutorial do BoTorch

In [1]:
import sys
import math
import os
import warnings
from dataclasses import dataclass
from typing import Optional

import gpytorch
import torch
from gpytorch.constraints import Interval
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood
from torch.quasirandom import SobolEngine

from botorch.acquisition import qExpectedImprovement, qLogExpectedImprovement
from botorch.exceptions import BadInitialCandidatesWarning
from botorch.fit import fit_gpytorch_mll
from botorch.generation import MaxPosteriorSampling
from botorch.models import SingleTaskGP
from botorch.optim import optimize_acqf
from botorch.test_functions import Ackley
from botorch.utils.transforms import unnormalize


warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

In [2]:
fun = Ackley(dim=20, negate=True).to(dtype=dtype, device=device)
fun.bounds[0, :].fill_(-5)
fun.bounds[1, :].fill_(10)
dim = fun.dim
lb, ub = fun.bounds

batch_size = 4
n_init = 2 * dim
max_cholesky_size = float("inf")  # Always use Cholesky


def eval_objective(x: torch.Tensor) -> torch.Tensor:
    """This is a helper function we use to unnormalize and evalaute a point."""
    return fun(unnormalize(x, fun.bounds))

In [3]:
@dataclass
class TurboState:
    """Turbo state used to track the recent history of the trust region."""
    dim: int
    batch_size: int
    length: float = 0.8
    length_min: float = 0.5**7
    length_max: float = 1.6
    failure_counter: int = 0
    failure_tolerance: int = float("nan")  # Note: Post-initialized
    success_counter: int = 0
    success_tolerance: int = 10  # Note: The original paper uses 3
    best_value: float = -float("inf")
    restart_triggered: bool = False

    def __post_init__(self):
        """Post-initialize the state of the trust region."""
        self.failure_tolerance = math.ceil(
            max([4.0 / self.batch_size, float(self.dim) / self.batch_size])
        )


def update_state(state: TurboState, Y_next: torch.Tensor) -> TurboState:
    """Update the state of the trust region based on the new function values."""
    if max(Y_next) > state.best_value + 1e-3 * math.fabs(state.best_value):
        state.success_counter += 1
        state.failure_counter = 0
    else:
        state.success_counter = 0
        state.failure_counter += 1

    if state.success_counter == state.success_tolerance:  # Expand trust region
        state.length = min(2.0 * state.length, state.length_max)
        state.success_counter = 0
    elif state.failure_counter == state.failure_tolerance:  # Shrink trust region
        state.length /= 2.0
        state.failure_counter = 0

    state.best_value = max(state.best_value, max(Y_next).item())
    if state.length < state.length_min:
        state.restart_triggered = True
    return state

In [4]:
state = TurboState(dim=dim, batch_size=batch_size)
print(state)

TurboState(dim=20, batch_size=4, length=0.8, length_min=0.0078125, length_max=1.6, failure_counter=0, failure_tolerance=5, success_counter=0, success_tolerance=10, best_value=-inf, restart_triggered=False)


In [5]:
def get_initial_points(dim: int, n_pts: int, seed: int = 0) -> torch.Tensor:
    """Generate initial points using Sobol sequence."""
    sobol = SobolEngine(dimension=dim, scramble=True, seed=seed)
    return sobol.draw(n=n_pts).to(dtype=dtype, device=device)

In [6]:
def generate_batch(
    state: TurboState,
    model: SingleTaskGP,  # GP model
    X: torch.Tensor,  # Evaluated points on the domain [0, 1]^d
    Y: torch.Tensor,  # Function values
    batch_size: int,
    n_candidates: Optional[int] = None,  # Number of candidates for Thompson sampling
    num_restarts: int = 10,
    raw_samples: int = 512,
    acqf: str = "ts",  # "ei" or "ts"
) -> torch.Tensor:
    """Generate a new batch of points."""
    assert acqf in ("ts", "ei")
    assert X.min() >= 0.0
    assert X.max() <= 1.0
    assert torch.all(torch.isfinite(Y))
    if n_candidates is None:
        n_candidates = min(5000, max(2000, 200 * X.shape[-1]))

    # Scale the TR to be proportional to the lengthscales
    x_center = X[Y.argmax(), :].clone()
    weights = model.covar_module.base_kernel.lengthscale.squeeze().detach()
    weights = weights / weights.mean()
    weights = weights / torch.prod(weights.pow(1.0 / len(weights)))
    tr_lb = torch.clamp(x_center - weights * state.length / 2.0, 0.0, 1.0)
    tr_ub = torch.clamp(x_center + weights * state.length / 2.0, 0.0, 1.0)

    if acqf == "ts":
        dim = X.shape[-1]
        sobol = SobolEngine(dim, scramble=True)
        pert = sobol.draw(n_candidates).to(dtype=dtype, device=device)
        pert = tr_lb + (tr_ub - tr_lb) * pert

        # Create a perturbation mask
        prob_perturb = min(20.0 / dim, 1.0)
        mask = torch.rand(n_candidates, dim, dtype=dtype, device=device) <= prob_perturb
        ind = torch.where(mask.sum(dim=1) == 0)[0]
        mask[ind, torch.randint(0, dim - 1, size=(len(ind),), device=device)] = 1

        # Create candidate points from the perturbations and the mask
        X_cand = x_center.expand(n_candidates, dim).clone()
        X_cand[mask] = pert[mask]

        # Sample on the candidate points
        thompson_sampling = MaxPosteriorSampling(model=model, replacement=False)
        with torch.no_grad():  # We don't need gradients when using TS
            X_next = thompson_sampling(X_cand, num_samples=batch_size)

    elif acqf == "ei":
        ei = qExpectedImprovement(model, Y.max())
        X_next, acq_value = optimize_acqf(
            ei,
            bounds=torch.stack([tr_lb, tr_ub]),
            q=batch_size,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )

    return X_next

In [7]:
X_turbo = get_initial_points(dim, n_init)
Y_turbo = torch.tensor(
    [eval_objective(x) for x in X_turbo], dtype=dtype, device=device
).unsqueeze(-1)

state = TurboState(dim, batch_size=batch_size, best_value=max(Y_turbo).item())

NUM_RESTARTS = 10 if not SMOKE_TEST else 2
RAW_SAMPLES = 512 if not SMOKE_TEST else 4
N_CANDIDATES = min(5000, max(2000, 200 * dim)) if not SMOKE_TEST else 4

torch.manual_seed(0)

while not state.restart_triggered:  # Run until TuRBO converges
    # Fit a GP model
    train_Y = (Y_turbo - Y_turbo.mean()) / Y_turbo.std()
    likelihood = GaussianLikelihood(noise_constraint=Interval(1e-8, 1e-3))
    covar_module = ScaleKernel(  # Use the same lengthscale prior as in the TuRBO paper
        MaternKernel(nu=2.5, ard_num_dims=dim, lengthscale_constraint=Interval(0.005, 4.0))
    )
    model = SingleTaskGP(X_turbo, train_Y, covar_module=covar_module, likelihood=likelihood)
    mll = ExactMarginalLogLikelihood(model.likelihood, model)

    # Do the fitting and acquisition function optimization inside the Cholesky context
    with gpytorch.settings.max_cholesky_size(max_cholesky_size):
        # Fit the model
        fit_gpytorch_mll(mll)

        # Create a batch
        X_next = generate_batch(
            state=state,
            model=model,
            X=X_turbo,
            Y=train_Y,
            batch_size=batch_size,
            n_candidates=N_CANDIDATES,
            num_restarts=NUM_RESTARTS,
            raw_samples=RAW_SAMPLES,
            acqf="ts",
        )

    Y_next = torch.tensor(
        [eval_objective(x) for x in X_next], dtype=dtype, device=device
    ).unsqueeze(-1)

    # Update state
    state = update_state(state=state, Y_next=Y_next)

    # Append data
    X_turbo = torch.cat((X_turbo, X_next), dim=0)
    Y_turbo = torch.cat((Y_turbo, Y_next), dim=0)

    # Print current status
    print(f"{len(X_turbo)}) Best value: {state.best_value:.2e}, TR length: {state.length:.2e}")

44) Best value: -1.17e+01, TR length: 8.00e-01
48) Best value: -1.17e+01, TR length: 8.00e-01
52) Best value: -1.09e+01, TR length: 8.00e-01
56) Best value: -9.91e+00, TR length: 8.00e-01
60) Best value: -9.91e+00, TR length: 8.00e-01
64) Best value: -9.91e+00, TR length: 8.00e-01
68) Best value: -9.25e+00, TR length: 8.00e-01
72) Best value: -9.25e+00, TR length: 8.00e-01
76) Best value: -9.25e+00, TR length: 8.00e-01
80) Best value: -9.25e+00, TR length: 8.00e-01
84) Best value: -8.66e+00, TR length: 8.00e-01
88) Best value: -8.13e+00, TR length: 8.00e-01
92) Best value: -8.13e+00, TR length: 8.00e-01
96) Best value: -8.13e+00, TR length: 8.00e-01
100) Best value: -7.84e+00, TR length: 8.00e-01
104) Best value: -7.49e+00, TR length: 8.00e-01
108) Best value: -7.49e+00, TR length: 8.00e-01
112) Best value: -7.49e+00, TR length: 8.00e-01
116) Best value: -7.49e+00, TR length: 8.00e-01
120) Best value: -7.49e+00, TR length: 8.00e-01
124) Best value: -7.49e+00, TR length: 4.00e-01
128) B